# GolStats — Machine Learning: Player Clustering

## Objective
Group players into style-based clusters using their tournament-level
statistics, without predefined labels. The goal is to discover natural
player profiles (e.g. finishers, playmakers, pressers) purely from the
data, using K-Means clustering.

## Source table
`golstats.gold.player_tournament`

## Approach
1. Filter to players with meaningful playing time.
2. Engineer per-match rate features (to avoid bias toward players who
   played more matches).
3. Scale features and run K-Means.
4. Interpret and label each cluster based on its dominant statistics.
5. Persist the result as a new Gold table for the dashboard.

In [0]:
from pyspark.sql.functions import col

df = spark.table("golstats.gold.player_tournament")

# Filtramos jugadores con muy pocos minutos — meten ruido en el clustering
df = df.filter(col("matches_played") >= 3)

# Convertimos todo a "tasas por partido" para que un jugador que jugó
# 7 partidos no aparezca artificialmente "más activo" que uno con 3
df_features = df.select(
    "player_id",
    "player",
    "matches_played",
    (col("goals") / col("matches_played")).alias("goals_per_match"),
    (col("xg") / col("matches_played")).alias("xg_per_match"),
    (col("shots") / col("matches_played")).alias("shots_per_match"),
    (col("passes") / col("matches_played")).alias("passes_per_match"),
    "pass_completion_pct",
    (col("pressures") / col("matches_played")).alias("pressures_per_match"),
    (col("ball_recoveries") / col("matches_played")).alias("recoveries_per_match"),
)

display(df_features.head(10))

In [0]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans

feature_cols = [
    "goals_per_match",
    "xg_per_match",
    "shots_per_match",
    "passes_per_match",
    "pass_completion_pct",
    "pressures_per_match",
    "recoveries_per_match",
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw")
df_vector = assembler.transform(df_features)

scaler = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True)
scaler_model = scaler.fit(df_vector)
df_scaled = scaler_model.transform(df_vector)

kmeans = KMeans(featuresCol="features", predictionCol="cluster", k=4, seed=42)
model = kmeans.fit(df_scaled)
df_clustered = model.transform(df_scaled)

display(df_clustered.select("player", "matches_played", *feature_cols, "cluster").head(10))

## Interpreting the clusters

K-Means groups players by statistical similarity, but the cluster
numbers themselves are meaningless — we need to inspect the average
profile of each cluster to understand what kind of player it represents.

In [0]:
from pyspark.sql.functions import avg, count

cluster_profiles = (
    df_clustered.groupBy("cluster")
    .agg(
        count("*").alias("n_players"),
        avg("goals_per_match").alias("avg_goals"),
        avg("xg_per_match").alias("avg_xg"),
        avg("shots_per_match").alias("avg_shots"),
        avg("passes_per_match").alias("avg_passes"),
        avg("pass_completion_pct").alias("avg_pass_pct"),
        avg("pressures_per_match").alias("avg_pressures"),
        avg("recoveries_per_match").alias("avg_recoveries"),
    )
    .orderBy("cluster")
)

display(cluster_profiles)

## Labeling the clusters

Based on the average profile of each cluster, we assign a human-readable
label. These labels are inferred purely from statistical behavior — we
don't have player position in this table, so they describe playing style,
not confirmed position.

- Cluster 2 → Finishers (highest goals, xG, shots)
- Cluster 0 → High-volume engines (most passes, pressures, recoveries)
- Cluster 3 → Clean possession, low defensive engagement
- Cluster 1 → Low involvement

In [0]:
from pyspark.sql.functions import when

df_labeled = df_clustered.withColumn(
    "player_profile",
    when(col("cluster") == 2, "Finisher")
    .when(col("cluster") == 0, "High-volume engine")
    .when(col("cluster") == 3, "Clean possession, low defensive engagement")
    .when(col("cluster") == 1, "Low involvement")
)

df_labeled.select(
    "player_id", "player", "matches_played", *feature_cols, "cluster", "player_profile"
).write.format("delta").mode("overwrite").saveAsTable("golstats.gold.player_clusters")

display(df_labeled.select("player", "player_profile", "cluster").orderBy("cluster"))

## Limitation found: goalkeepers cluster with low-minute players

Cluster 1 ("Low involvement") mixes two structurally different groups:
players with genuinely limited playing time, and goalkeepers — whose
event profile (few shots, passes, pressures in open play) looks
statistically similar to a bench player, even though their role is
completely different. This happens because player position is not
available as a feature. A future improvement would be to add position
and either exclude goalkeepers from this clustering or model them
separately, since their statistical profile is not comparable to
outfield players.

In [0]:
from pyspark.ml.evaluation import ClusteringEvaluator

evaluator = ClusteringEvaluator(
    featuresCol="features",
    predictionCol="cluster",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean"
)
score = evaluator.evaluate(df_clustered)
print(f"Silhouette score: {score:.3f}")

## Silhouette score interpretation

Silhouette score: 0.271

This is a moderate score — clusters are meaningfully separated but with
some overlap, which is consistent with the limitation found above
(goalkeepers mixed into the "Low involvement" cluster due to missing
position data). A higher score could likely be achieved by excluding
goalkeepers or adding player position as a feature, but the current
clustering already produced interpretable, football-realistic groups
(finishers, midfield engines, low-involvement players), so no further
tuning was done at this stage.

## Predicting player goals from underlying metrics

Objective: predict a player's total goals in the tournament using their
underlying volume and quality metrics (xG, shots, passes, pressures,
recoveries, matches played). Using raw totals here (not per-match rates),
since we're predicting a total, not a rate.

Model: Linear Regression, 80/20 train/test split.

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

df_reg = spark.table("golstats.gold.player_tournament").filter(col("matches_played") >= 3)

reg_features = ["xg", "shots", "passes", "completed_passes", "pressures", "ball_recoveries", "matches_played"]

assembler = VectorAssembler(inputCols=reg_features, outputCol="features")
df_reg_vector = assembler.transform(df_reg).select("player", "features", "goals")

train_df, test_df = df_reg_vector.randomSplit([0.8, 0.2], seed=42)

lr = LinearRegression(featuresCol="features", labelCol="goals")
lr_model = lr.fit(train_df)

predictions = lr_model.transform(test_df)

evaluator_rmse = RegressionEvaluator(labelCol="goals", predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="goals", predictionCol="prediction", metricName="r2")

rmse = evaluator_rmse.evaluate(predictions)
r2 = evaluator_r2.evaluate(predictions)

print(f"RMSE: {rmse:.3f}")
print(f"R²: {r2:.3f}")

display(predictions.select("player", "goals", "prediction"))

## Model evaluation: negative R²

RMSE: 0.541 | R²: -0.054

The negative R² indicates the model performs worse than predicting the
mean for every player. This is expected given the nature of the target:
goals are sparse count data (most players score 0), which violates the
Gaussian error assumption behind Linear Regression. The small test set
size (~57 players) also makes R² unstable. In absolute terms, the RMSE
(0.541 goals) is reasonably low — but R² is the more honest metric here,
and it shows Linear Regression is not the right tool for this target.

Next: try a model built for count data instead — Poisson regression.

In [0]:
from pyspark.ml.regression import GeneralizedLinearRegression

glr = GeneralizedLinearRegression(featuresCol="features", labelCol="goals", family="poisson", link="log")
glr_model = glr.fit(train_df)

predictions_poisson = glr_model.transform(test_df)

rmse_poisson = evaluator_rmse.evaluate(predictions_poisson)
r2_poisson = evaluator_r2.evaluate(predictions_poisson)

print(f"Poisson RMSE: {rmse_poisson:.3f}")
print(f"Poisson R²: {r2_poisson:.3f}")

display(predictions_poisson.select("player", "goals", "prediction"))

## Model comparison: Linear Regression vs Poisson Regression

| Model              | RMSE  | R²     |
|---------------------|-------|--------|
| Linear Regression    | 0.541 | -0.054 |
| Poisson Regression    | 0.496 | 0.115  |

Poisson regression outperforms Linear Regression on both metrics, as
expected — goals are count data, and Poisson regression models that
distribution directly instead of assuming continuous, normally
distributed errors.

The R² of 0.115 is still modest: predicting individual player goals from
a single tournament is inherently hard, since goal-scoring has a large
random component (a single deflected shot, a matchup against a weak
goalkeeper) that no set of underlying stats fully captures. The model
correctly identifies directional signal — for example, Randal Kolo Muani
(1 actual goal) and Nikola Vlašić (0 goals) both receive high predicted
values because of strong underlying shot/xG volume — but exact goal
counts remain hard to pin down with only ~250 data points.

This is presented as an honest, appropriately-scoped result rather than
an overstated one: the value here is in comparing modeling approaches
correctly for the data type, not in claiming high predictive accuracy.

In [0]:
predictions_poisson.select("player", "goals", "prediction").write.format("delta").mode("overwrite").saveAsTable("golstats.gold.goal_predictions")